### vllm rollout engine

- https://verl.readthedocs.io/en/latest/perf/perf_tuning.html
- https://docs.vllm.ai/en/latest/configuration/optimization.html
- max_num_seqs vs. max_num_batched_tokens vs. max_model_len
    - max_model_len：模型的最大生成长度，包含prompt长度和generated长度。（默认为 model 的 context size）
    - max_num_batched_tokens: 定义了在一次模型推理（或称为一个批次）中，可以处理的最大 token 数量。
        - 它把该步内所有活跃序列要处理的 token（包含新请求的 prefill token 以及老请求的 decode token）合在一起计数
- verl

```
vllm serve meta-llama/Meta-Llama-3-8B-Instruct \
  --gpu-memory-utilization 0.90 \
  --max-model-len 8192 \
  --max-num-batched-tokens 4096 \
  --max-num-seqs 256 \
  --long-prefill-token-threshold 1024 \
  --max-num-partial-prefills 2 \
  --max-long-partial-prefills 1
```
- 假设同一时刻到来 5 个请求（都设 max_new_tokens=200）：
    - R1：prompt=6000 tok（很长，会被分块）
    - R2、R3、R4：prompt=200 tok（短）
    - R5：prompt=1500 tok（也会被分块）
- 我们用上面的配置（4096 总预算；长 prefill 每块 1024）。调度器每一步会装配一个“混合批次”，满足
    - Σ(本步 prefill 块 token) + Σ(本步 decode token) ≤ 4096 且 序列数 ≤ 256。

- step 1(全是 prefill 起步）
    - 给 R1 一块 1024（因阈值 1024）
    - R2、R3、R4 各自整个 prompt 200×3=600（它们不够“长”，无需再切）
    - 给 R5 一块 1024
    - 本步总计：1024 + 600 + 1024 = 2648 ≤ 4096
        - 结果：R2/R3/R4 的 prefill 完成，进入 decode 阶段；R1 余 4976，R5 余 476。（vLLM 会把 prefill 与 decode 放在迭代级混合批里，以提高利用率。）
- Step 2（prefill + decode 混跑）
    - R1 再给一块 1024
    - R5 把剩余 476 一次吃完
    - 同时对 R2/R3/R4 做 decode：每条各 1 个新 token ⇒ 3
    - 本步总计：1024 + 476 + 3 = 1503 ≤ 4096
        - 结果：R5 进入 decode；R2/R3/R4 已有 1 个输出 token；R1 余 3952。
- Step 3（继续混跑）
    - R1：1024
    - R2/R3/R4/R5：decode 4
    - 本步总计：1028；R1 余 2928。
- Step 4
    - R1：1024；4 个 decode ⇒ 4
    - 本步总计：1028；R1 余 1904。
- Step 5
    - R1：1024；4 个 decode ⇒ 4
    - 本步总计：1028；R1 余 880。
- Step 6（R1 收尾）
    - R1：把余下 880 吃完；4 个 decode ⇒ 4
    - 本步总计：884；至此 所有请求都在 decode。